In [1]:
# Core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import MSELoss
# Data handling
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader,TensorDataset

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics & utilities
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Progress & debugging
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import glob
import joblib

device = "mps" if torch.backends.mps.is_available() else "cpu"


In [2]:
all_files = sorted(glob.glob("data/monthly_hourly_load_values_*.xlsx"))
df_list = [pd.read_excel(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)
df['DateUTC'] = pd.to_datetime(df['DateUTC'])
df.set_index('DateUTC', inplace=True)

print(f"Dataframe shape: {df.shape}")
# # Training: 2019–2024, Testing: 2025
# train_df_full = df[df.index.year < 2025]
# test_df = df[df.index.year == 2025]

# # Split validation from end of 2024 (10% of training)
# val_ratio = 0.1
# val_size = int(len(train_df_full) * val_ratio)
# val_df = train_df_full.iloc[-val_size:]
# train_df = train_df_full.iloc[:-val_size]

# def create_sequences(data, seq_len=24, output_len=1):
#     X, y = [], []
#     for i in range(seq_len, len(data) - output_len + 1):
#         X.append(data[i - seq_len:i])
#         y.append(data[i:i + output_len].flatten())
#     return np.array(X), np.array(y)

# scaler = MinMaxScaler()
# scaler.fit(df[['Value']])
# joblib.dump(scaler, 'scaler_minmax.pkl')
# train_df['Value_normalized'] = scaler.transform(train_df[['Value']])
# val_df['Value_normalized']   = scaler.transform(val_df[['Value']])
# test_df['Value_normalized'] = scaler.transform(test_df[['Value']])


# def prepare_multi_country_data_per_country_scaler(df, sequence_length=24, prediction_length=1, dataset_name="train/val"):
#     """
#     For each country, fit a StandardScaler, save it, and normalize values.
#     """
#     print(f"\n{'='*50}")
#     print(f"PREPARING {dataset_name.upper()} WITH PER-COUNTRY SCALER")
#     print(f"{'='*50}")

#     df_clean = df.dropna(subset=['Value']).copy()
#     df_clean = df_clean.sort_values(['CountryCode', 'DateUTC']).reset_index(drop=False)

#     sequences = []
#     targets = []
#     countries = []
#     timestamps = []

#     for country in df_clean['CountryCode'].unique():
#         country_data = df_clean[df_clean['CountryCode'] == country]
#         scaler = StandardScaler()
#         scaler.fit(country_data[['Value']])
#         # Save scaler for later use
#         scaler_filename = f"scaler_{country}.pkl"
#         joblib.dump(scaler, scaler_filename)
#         # Normalize values
#         country_data['Value_normalized'] = scaler.transform(country_data[['Value']])
#         values = country_data['Value_normalized'].values
#         dates = country_data['DateUTC'].values
#         X_seq, y_seq = create_sequences(values, seq_len=sequence_length, output_len=prediction_length)
#         sequences.append(X_seq)
#         targets.append(y_seq)
#         countries.extend([country] * len(X_seq))
#         timestamps.extend(dates[sequence_length:sequence_length+len(X_seq)])

#     X = np.concatenate(sequences, axis=0) if sequences else np.array([])
#     y = np.concatenate(targets, axis=0) if targets else np.array([])
#     countries = np.array(countries)
#     timestamps = np.array(timestamps)

#     print(f"\nFinal dataset stats:")
#     print(f"Total sequences: {len(X):,}")
#     print(f"X shape: {X.shape}")
#     print(f"y shape: {y.shape}")
#     print(f"Normalized X range: [{X.min():.3f}, {X.max():.3f}]")
#     print(f"Normalized y range: [{y.min():.3f}, {y.max():.3f}]")

#     unique_countries, counts = np.unique(countries, return_counts=True)
#     print(f"\nCountry distribution in sequences:")
#     for country, count in zip(unique_countries, counts):
#         print(f"  {country}: {count:,} sequences ({count/len(X)*100:.1f}%)")

#     return X, y

# X_train, y_train = prepare_multi_country_data_per_country_scaler(
#     train_df, sequence_length=24, prediction_length=1
# )

# X_val, y_val= prepare_multi_country_data_per_country_scaler(
#     val_df, sequence_length=24, prediction_length=1
# )

# X_test, y_test = prepare_multi_country_data_per_country_scaler(
#     test_df, sequence_length=24, prediction_length=1,dataset_name="test"
# )

# print(f"All datasets normalized with the same scaler")
# print(f"Train range: [{X_train.min():.3f}, {X_train.max():.3f}]")
# print(f"Val range:   [{X_val.min():.3f}, {X_val.max():.3f}]")
# print(f"Test range:  [{X_test.min():.3f}, {X_test.max():.3f}]")




Dataframe shape: (2047167, 10)
